> **교육 목표** 원료 변경으로 데이터 관계가 바뀌는 개념 드리프트를 이해하고, 고정 모델과 온라인 러닝을 비교합니다.
>
> **핵심 내용** 배합 기록을 30건씩 흘려보내기 → 고정 모델 vs `partial_fit` 갱신 모델 → MAE·평균오차 추이 그래프

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D4_04 드리프트와 온라인 러닝

> **오늘 질문:** 원료가 바뀌어 강도가 떨어지기 시작하면, 모델은 언제 알아채고 어떻게 따라갈까?

**실험 설정**
- 콘크리트 배합 기록이 하나씩 순서대로 들어온다고 가정합니다.
- 처음 500건으로 모델을 만들고, 나머지 530건을 30건씩 흘려보냅니다.
- **650번째 기록부터 원료가 바뀌어** 같은 배합이라도 강도가 10 MPa 낮게 나옵니다(개념 드리프트).
- 고정 모델(처음 그대로) vs 온라인 모델(30건마다 `partial_fit`으로 갱신)을 비교합니다.

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: 데이터 흐름과 드리프트 만들기 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False   # 음수 부호 깨짐 방지
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_absolute_error

cc = pd.read_csv("../../data/concrete.csv").sample(frac=1, random_state=42).reset_index(drop=True)
cc.loc[650:, "strength"] = cc.loc[650:, "strength"] - 10    # 650번째부터 원료 변경: 강도 10 MPa 하락
X = cc.drop(columns="strength")
y = cc["strength"]

# 처음 500건: 초기 학습용. 스케일러 기준도 여기서만 정하고 끝까지 그대로 쓴다
scaler = StandardScaler().fit(X.iloc[:500])
X_init = scaler.transform(X.iloc[:500])
y_init = y.iloc[:500]

## 1. 두 모델 초기 학습 ✍️

`SGDRegressor`는 조금씩 배우는 선형회귀입니다. `partial_fit`이 있어서 온라인 러닝이 됩니다. 두 모델은 처음에 똑같이 학습합니다.

> 변수 이름: `fixed`, `online`

In [ ]:
# 힌트: SGDRegressor(learning_rate="constant", eta0=0.01, random_state=42)

## 2. 데이터 흘려보내기 📋

30건이 들어올 때마다 ① 두 모델로 예측 → ② 실제 강도로 오차 기록 → ③ 온라인 모델만 `partial_fit`으로 갱신합니다.

In [ ]:
rows = []
for start in range(500, len(cc), 30):
    X_new = scaler.transform(X.iloc[start:start + 30])    # 새로 들어온 30건
    y_new = y.iloc[start:start + 30]

    pred_fixed = fixed.predict(X_new)
    pred_online = online.predict(X_new)
    rows.append({"시작": start,
                 "고정 MAE": mean_absolute_error(y_new, pred_fixed),
                 "온라인 MAE": mean_absolute_error(y_new, pred_online),
                 "고정 평균오차": (y_new - pred_fixed).mean(),
                 "온라인 평균오차": (y_new - pred_online).mean()})

    online.partial_fit(X_new, y_new)                       # 온라인 모델만 새 데이터로 조금 더 배운다
stream = pd.DataFrame(rows).set_index("시작")
stream.round(1)

## 3. MAE 추이 그래프 ✍️

회색 점선이 원료가 바뀐 시점(650번째)입니다.

In [ ]:
# 힌트: 표[["컬럼1", "컬럼2"]].plot(marker="o"), plt.axvline(위치)

## 4. 평균오차(치우침) 그래프 ✍️

평균오차 = 실제 − 예측의 평균. 0에서 멀어지면 모델이 한쪽으로 계속 틀린다는 뜻입니다.

## 정리 💬

**Q1.** 원료 변경 이후 두 모델의 MAE와 평균오차는 어떻게 달라졌나요?

> 답:


**Q2.** 온라인 러닝을 현장에서 쓴다면 어떤 장치가 함께 있어야 할까요? 이 실험의 '평균오차' 그래프는 무엇과 닮았나요?

> 답:
